First, I will load the dataset from GBIF

In [1]:
import pandas as pd
data = pd.read_csv("./germplasm_banks/occurrence.txt", sep= "\t")
print(list(data.columns))

['gbifID', 'accessRights', 'bibliographicCitation', 'language', 'license', 'modified', 'publisher', 'references', 'rightsHolder', 'type', 'institutionID', 'collectionID', 'datasetID', 'institutionCode', 'collectionCode', 'datasetName', 'ownerInstitutionCode', 'basisOfRecord', 'informationWithheld', 'dataGeneralizations', 'dynamicProperties', 'occurrenceID', 'catalogNumber', 'recordNumber', 'recordedBy', 'recordedByID', 'individualCount', 'organismQuantity', 'organismQuantityType', 'sex', 'lifeStage', 'reproductiveCondition', 'caste', 'behavior', 'vitality', 'establishmentMeans', 'degreeOfEstablishment', 'pathway', 'georeferenceVerificationStatus', 'occurrenceStatus', 'preparations', 'disposition', 'associatedOccurrences', 'associatedReferences', 'associatedSequences', 'associatedTaxa', 'otherCatalogNumbers', 'occurrenceRemarks', 'organismID', 'organismName', 'organismScope', 'associatedOrganisms', 'previousIdentifications', 'organismRemarks', 'materialEntityID', 'materialEntityRemarks'

# Germplasm
First, I do the mapping from GBIF's data to our CSV template.
As we only have data from GBIF at this point, all of the taxonomy reference column is filled with GBIF's taxonomy reference.
To create the contents of the iucn_threat_category cell, I have to add the rest of the URI to the category that is present in GBIF's dataset (e.g.: VU becomes http://rs.gbif.org/vocabulary/iucn/threat_status/VU)

In [2]:
germplasm = pd.DataFrame(columns=["uniqid", "accession_number", "national_catalogue_code", "taxon_identifier", "scientific_name", "scientific_name_authorship", "vernacular_name", "taxonomy_reference", "determination_status", "iucn_threat_category"])
germplasm[["accession_number", "taxon_identifier", "scientific_name", "vernacular_name"]] = data[["occurrenceID", "taxonID", "acceptedScientificName", "vernacularName"]]
germplasm["taxonomy_reference"] = "GBIF Secretariat: GBIF Backbone Taxonomy. https://doi.org/10.15468/39omei"
germplasm["iucn_threat_category"] = "http://rs.gbif.org/vocabulary/iucn/threat_status/" + data["iucnRedListCategory"]
germplasm

,uniqid,accession_number,national_catalogue_code,taxon_identifier,scientific_name,scientific_name_authorship,vernacular_name,taxonomy_reference,determination_status,iucn_threat_category
0,NaN,JBCLM:BGJBCLM:CM00013-08,NaN,NaN,Ziziphora aragonensis Pau,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
1,NaN,JBCLM:BGJBCLM:CM00009-08,NaN,NaN,Vella pseudocytisus subsp. pseudocytisus,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
2,NaN,JBCLM:BGJBCLM:CM00247-09,NaN,NaN,Zygophyllum fabago L.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
3,NaN,JBCLM:BGJBCLM:CM00854-13,NaN,NaN,Viburnum lantana L.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,http://rs.gbif.org/vocabulary/iucn/threat_stat...
4,NaN,JBCLM:BGJBCLM:CM00002-08,NaN,NaN,Ziziphora aragonensis Pau,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
903,NaN,JBCLM:BGJBCLM:CM00686-11,NaN,NaN,Anarrhinum fruticosum Desf.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
904,NaN,JBCLM:BGJBCLM:CM00243-09,NaN,NaN,Agrostemma githago L.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
905,NaN,JBCLM:BGJBCLM:CM00591-11,NaN,NaN,Adonis vernalis L.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
906,NaN,JBCLM:BGJBCLM:CM00629-11,NaN,NaN,Anarrhinum laxiflorum Boiss.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN


In [3]:
# Transform usageKey: Convert to Int64 and add URL prefix
def format_gbif_url(taxon_id):
    if pd.isna(taxon_id):
        return "N/A"
    try:
        return f"https://www.gbif.org/species/{int(taxon_id)}"
    except (ValueError, TypeError):
        return "N/A"

if germplasm["taxon_identifier"].isna().all() or (germplasm["taxon_identifier"] == "").all():
   germplasm["taxon_identifier"] = data["acceptedTaxonKey"].apply(format_gbif_url)
germplasm

,uniqid,accession_number,national_catalogue_code,taxon_identifier,scientific_name,scientific_name_authorship,vernacular_name,taxonomy_reference,determination_status,iucn_threat_category
0,NaN,JBCLM:BGJBCLM:CM00013-08,NaN,https://www.gbif.org/species/3899518,Ziziphora aragonensis Pau,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
1,NaN,JBCLM:BGJBCLM:CM00009-08,NaN,https://www.gbif.org/species/7225676,Vella pseudocytisus subsp. pseudocytisus,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
2,NaN,JBCLM:BGJBCLM:CM00247-09,NaN,https://www.gbif.org/species/3189902,Zygophyllum fabago L.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
3,NaN,JBCLM:BGJBCLM:CM00854-13,NaN,https://www.gbif.org/species/2888615,Viburnum lantana L.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,http://rs.gbif.org/vocabulary/iucn/threat_stat...
4,NaN,JBCLM:BGJBCLM:CM00002-08,NaN,https://www.gbif.org/species/3899518,Ziziphora aragonensis Pau,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
903,NaN,JBCLM:BGJBCLM:CM00686-11,NaN,https://www.gbif.org/species/8267849,Anarrhinum fruticosum Desf.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
904,NaN,JBCLM:BGJBCLM:CM00243-09,NaN,https://www.gbif.org/species/3085368,Agrostemma githago L.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
905,NaN,JBCLM:BGJBCLM:CM00591-11,NaN,https://www.gbif.org/species/5371711,Adonis vernalis L.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
906,NaN,JBCLM:BGJBCLM:CM00629-11,NaN,https://www.gbif.org/species/3742445,Anarrhinum laxiflorum Boiss.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN


Then, I will run the Taxonomic Name Resolution Service's (TNRS) API to transform the names to the World Flora Online Taxonomy. To do so, I first need to create a csv with IDs for each scientific name, as it is a requirement from the API

In [4]:
df2 = pd.DataFrame({
    "sciname": germplasm["scientific_name"].dropna().drop_duplicates()
})
df2.to_csv("./germplasm_banks/WFOAPI_input.csv", index = True, index_label="ID")

Then, I run the curl command to send my data to the API. The arguments that I've chosen are:
- -f: input file
- -o: output file
- -m: api mode. In this case, I've chosen resolve because it parses, matches to a published name and resolves names to accepted name.
- -s: taxonomic sources. In this case I only want to use WFO as taxonomic source.
- -k: keep files. Writes input and output as CSV files.

I had to modify the script that was provided to use the API (tnrsapi.sh), as it was passing all scientific names to be resolved as arguments in the curl command, exceeding the limit of characters in a command. I changed it to load from a file.

In [5]:
import subprocess
import os
script_path = "./tnrsapi.sh"
args = [
    "-f", "/home/acb8/Work/FLAIR-GG_models/germplasm_banks/WFOAPI_input.csv",
    "-o", "/home/acb8/Work/FLAIR-GG_models/germplasm_banks/WFOAPI_input_resolved.csv",
    "-m", "resolve",
    "-s", "wfo",
    "-k"
]

try:
    result = subprocess.run(
        [script_path] + args,
        capture_output=True,
        text=True
    )
    print("Script output:")
    print(result.stdout)
    if result.stderr:
        print("Script errors:")
        print(result.stderr)
    if result.returncode == 0:
        print("Script ran successfully.")
        # Check if output CSV exists
        if os.path.exists("/home/acb8/Work/FLAIR-GG_models/germplasm_banks/WFOAPI_input_resolved.csv"):
            print("Output CSV generated successfully.")
        else:
            print("Output CSV not found.")
    else:
        print(f"Script failed with return code {result.returncode}.")
except FileNotFoundError:
    print("Error: tnrsapi.sh not found in current directory.")
except subprocess.CalledProcessError as e:
    print(f"Error running script: {e}")
    print(f"Errors: {e.stderr}")

Script output:
 
Names submitted:
|  ID | sciname                                                                                                     |
| --- | ----------------------------------------------------------------------------------------------------------- |
|   0 | Ziziphora aragonensis Pau                                                                                   |
|   1 | Vella pseudocytisus subsp. pseudocytisus                                                                    |
|   2 | Zygophyllum fabago L.                                                                                       |
|   3 | Viburnum lantana L.                                                                                         |
|   6 | Viola cazorlensis Gand.                                                                                     |
|   8 | Viburnum tinus L.                                                                                           |
|   9 | Veronica chama

Now I create a dataframe with the matches (dfWFO), and fill it with the results from the API matching. If there is an accepted name (i.e.: if the "Taxonomic_status" is Accepted or Synonym), that's what will be used, but if the "Taxonomic_status" is No opinion, then those columns are emtpy, so I will used the name that was matched from the WFO taxonomy. The same happens for the taxon identifier and authorship.

Note: there are less matches than rows because some are repeated

In [6]:
import numpy as np
dfWFO = pd.read_csv("/home/acb8/Work/FLAIR-GG_models/germplasm_banks/WFOAPI_input_resolved.csv")
dfWFO["scientific_name"] = np.where(
    dfWFO["Accepted_name"].notna() & (dfWFO["Accepted_name"] != ""),
    dfWFO["Accepted_name"],
    dfWFO["Name_matched"]
)
dfWFO["taxon_identifier"] = np.where(
    dfWFO["Accepted_name_url"].notna() & (dfWFO["Accepted_name_url"] != ""),
    dfWFO["Accepted_name_url"],
    dfWFO["Name_matched_url"]
)

dfWFO["scientific_name_authorship"] = np.where(
    dfWFO["Accepted_name_author"].notna() & (dfWFO["Accepted_name_author"] != ""),
    dfWFO["Accepted_name_author"],
    dfWFO["Canonical_author"]
)
dfWFO = dfWFO[["Name_submitted", "Author_submitted", "scientific_name", "taxon_identifier", "scientific_name_authorship"]]

dfWFO

,Name_submitted,Author_submitted,scientific_name,taxon_identifier,scientific_name_authorship
0,Ziziphora aragonensis Pau,Pau,Ziziphora aragonensis,https://www.worldfloraonline.org/taxon/wfo-000...,Pau
1,Vella pseudocytisus subsp. pseudocytisus,NaN,Vella pseudocytisus,https://www.worldfloraonline.org/taxon/wfo-000...,L.
2,Zygophyllum fabago L.,L.,Zygophyllum fabago,https://www.worldfloraonline.org/taxon/wfo-000...,L.
3,Viburnum lantana L.,L.,Viburnum lantana,https://www.worldfloraonline.org/taxon/wfo-000...,L.
4,Viola cazorlensis Gand.,Gand.,Viola cazorlensis,https://www.worldfloraonline.org/taxon/wfo-000...,Gand.
...,...,...,...,...,...
507,Aconitum napellus subsp. napellus,NaN,Aconitum napellus,https://www.worldfloraonline.org/taxon/wfo-000...,L.
508,Anarrhinum fruticosum Desf.,Desf.,Anarrhinum fruticosum,https://www.worldfloraonline.org/taxon/wfo-000...,Desf.
509,Agrostemma githago L.,L.,Agrostemma githago,https://www.worldfloraonline.org/taxon/wfo-000...,L.
510,Adonis vernalis L.,L.,Adonis vernalis,https://www.worldfloraonline.org/taxon/wfo-000...,L.


One of the benefits of the API is that it splits the name you submit from its authorship ("Name_submitted" and "Author_submitted" respectively). The issue is that it doesn't return ONLY the scientific name, so I'll remove the contents of the Author_submitted column from the Name_submitted column to get only the scientific name.

Now, to include the information from the germplasm banks in the entries of the new names, I have to match the "Name_submitted" column from dfWFO with "scientific_name" from the original gemrplasm df. The issue is that the API returns the Name_submitted column with the diacritics (non-ascii characters) and commas removed, so I have to do some preprocessing before merging both dataframes so that the germplasm["scientific_name"] and dfWFO["Name_submitted"] columns match.

In [7]:
print(list([germplasm["scientific_name"][0], dfWFO["Name_submitted"][0]]))

['Ziziphora aragonensis Pau', 'Ziziphora aragonensis Pau']


In [8]:
import unicodedata
# Function to remove diacritics
def remove_diacritics(text):
    if pd.isna(text):
        return text
    # Normalize to decomposed form (separate character and diacritic)
    normalized = unicodedata.normalize('NFD', str(text))
    # Remove non-ASCII characters (diacritics)
    return ''.join(c for c in normalized if unicodedata.category(c) != 'Mn').strip()
    
germplasm["temp_sci_name"] = germplasm["scientific_name"].apply(remove_diacritics)
#replace commas with space
germplasm["temp_sci_name"] = germplasm["temp_sci_name"].str.replace(',', ' ', regex=False)

In [9]:
dfWFO["Name_submitted"][5]

'Viburnum tinus L.'

In [10]:
dfWFO["Name_submitted"][5] == germplasm["temp_sci_name"].str.replace(',', ' ', regex=False)[15]

False

Now I'll merge both dataframes, using this temporary scientific name column (without the commas and diacritics) from the germplasm df and the "Name_submitted" column from the dfWFO df. If column names coincide, they will be tagged with their original dataframe to avoid confusion or loss of information

In [11]:
merged_df = pd.merge(germplasm, dfWFO, left_on = "temp_sci_name", right_on = "Name_submitted", how = "inner", suffixes = ("_germplasm", "_wfo"))
merged_df

,uniqid,accession_number,national_catalogue_code,taxon_identifier_germplasm,scientific_name_germplasm,scientific_name_authorship_germplasm,vernacular_name,taxonomy_reference,determination_status,iucn_threat_category,temp_sci_name,Name_submitted,Author_submitted,scientific_name_wfo,taxon_identifier_wfo,scientific_name_authorship_wfo
0,NaN,JBCLM:BGJBCLM:CM00013-08,NaN,https://www.gbif.org/species/3899518,Ziziphora aragonensis Pau,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN,Ziziphora aragonensis Pau,Ziziphora aragonensis Pau,Pau,Ziziphora aragonensis,https://www.worldfloraonline.org/taxon/wfo-000...,Pau
1,NaN,JBCLM:BGJBCLM:CM00009-08,NaN,https://www.gbif.org/species/7225676,Vella pseudocytisus subsp. pseudocytisus,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN,Vella pseudocytisus subsp. pseudocytisus,Vella pseudocytisus subsp. pseudocytisus,NaN,Vella pseudocytisus,https://www.worldfloraonline.org/taxon/wfo-000...,L.
2,NaN,JBCLM:BGJBCLM:CM00247-09,NaN,https://www.gbif.org/species/3189902,Zygophyllum fabago L.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN,Zygophyllum fabago L.,Zygophyllum fabago L.,L.,Zygophyllum fabago,https://www.worldfloraonline.org/taxon/wfo-000...,L.
3,NaN,JBCLM:BGJBCLM:CM00854-13,NaN,https://www.gbif.org/species/2888615,Viburnum lantana L.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,http://rs.gbif.org/vocabulary/iucn/threat_stat...,Viburnum lantana L.,Viburnum lantana L.,L.,Viburnum lantana,https://www.worldfloraonline.org/taxon/wfo-000...,L.
4,NaN,JBCLM:BGJBCLM:CM00002-08,NaN,https://www.gbif.org/species/3899518,Ziziphora aragonensis Pau,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN,Ziziphora aragonensis Pau,Ziziphora aragonensis Pau,Pau,Ziziphora aragonensis,https://www.worldfloraonline.org/taxon/wfo-000...,Pau
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
902,NaN,JBCLM:BGJBCLM:CM00686-11,NaN,https://www.gbif.org/species/8267849,Anarrhinum fruticosum Desf.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN,Anarrhinum fruticosum Desf.,Anarrhinum fruticosum Desf.,Desf.,Anarrhinum fruticosum,https://www.worldfloraonline.org/taxon/wfo-000...,Desf.
903,NaN,JBCLM:BGJBCLM:CM00243-09,NaN,https://www.gbif.org/species/3085368,Agrostemma githago L.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN,Agrostemma githago L.,Agrostemma githago L.,L.,Agrostemma githago,https://www.worldfloraonline.org/taxon/wfo-000...,L.
904,NaN,JBCLM:BGJBCLM:CM00591-11,NaN,https://www.gbif.org/species/5371711,Adonis vernalis L.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN,Adonis vernalis L.,Adonis vernalis L.,L.,Adonis vernalis,https://www.worldfloraonline.org/taxon/wfo-000...,L.
905,NaN,JBCLM:BGJBCLM:CM00629-11,NaN,https://www.gbif.org/species/3742445,Anarrhinum laxiflorum Boiss.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN,Anarrhinum laxiflorum Boiss.,Anarrhinum laxiflorum Boiss.,Boiss.,Anarrhinum laxiflorum,https://www.worldfloraonline.org/taxon/wfo-000...,Boiss.


In [12]:
merged_df = merged_df[["uniqid", "accession_number", "national_catalogue_code", "taxon_identifier_wfo", "scientific_name_wfo","scientific_name_authorship_wfo","vernacular_name", "taxonomy_reference", "determination_status", "iucn_threat_category"]]
merged_df["taxonomy_reference"] = "WFO (2025): World Flora Online. Published on the Internet; http://www.worldfloraonline.org"
merged_df.rename(columns={"taxon_identifier_wfo" : "taxon_identifier", "scientific_name_wfo" : "scientific_name", "scientific_name_authorship_wfo" : "scientific_name_authorship"}, inplace = True)
merged_df

/tmp/ipykernel_14499/411484739.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_df["taxonomy_reference"] = "WFO (2025): World Flora Online. Published on the Internet; http://www.worldfloraonline.org"
/tmp/ipykernel_14499/411484739.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_df.rename(columns={"taxon_identifier_wfo" : "taxon_identifier", "scientific_name_wfo" : "scientific_name", "scientific_name_authorship_wfo" : "scientific_name_authorship"}, inplace = True)


,uniqid,accession_number,national_catalogue_code,taxon_identifier,scientific_name,scientific_name_authorship,vernacular_name,taxonomy_reference,determination_status,iucn_threat_category
0,NaN,JBCLM:BGJBCLM:CM00013-08,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Ziziphora aragonensis,Pau,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
1,NaN,JBCLM:BGJBCLM:CM00009-08,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Vella pseudocytisus,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
2,NaN,JBCLM:BGJBCLM:CM00247-09,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Zygophyllum fabago,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
3,NaN,JBCLM:BGJBCLM:CM00854-13,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Viburnum lantana,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,http://rs.gbif.org/vocabulary/iucn/threat_stat...
4,NaN,JBCLM:BGJBCLM:CM00002-08,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Ziziphora aragonensis,Pau,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
902,NaN,JBCLM:BGJBCLM:CM00686-11,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Anarrhinum fruticosum,Desf.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
903,NaN,JBCLM:BGJBCLM:CM00243-09,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Agrostemma githago,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
904,NaN,JBCLM:BGJBCLM:CM00591-11,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Adonis vernalis,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
905,NaN,JBCLM:BGJBCLM:CM00629-11,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Anarrhinum laxiflorum,Boiss.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN


In [13]:
germplasm

,uniqid,accession_number,national_catalogue_code,taxon_identifier,scientific_name,scientific_name_authorship,vernacular_name,taxonomy_reference,determination_status,iucn_threat_category,temp_sci_name
0,NaN,JBCLM:BGJBCLM:CM00013-08,NaN,https://www.gbif.org/species/3899518,Ziziphora aragonensis Pau,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN,Ziziphora aragonensis Pau
1,NaN,JBCLM:BGJBCLM:CM00009-08,NaN,https://www.gbif.org/species/7225676,Vella pseudocytisus subsp. pseudocytisus,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN,Vella pseudocytisus subsp. pseudocytisus
2,NaN,JBCLM:BGJBCLM:CM00247-09,NaN,https://www.gbif.org/species/3189902,Zygophyllum fabago L.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN,Zygophyllum fabago L.
3,NaN,JBCLM:BGJBCLM:CM00854-13,NaN,https://www.gbif.org/species/2888615,Viburnum lantana L.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,http://rs.gbif.org/vocabulary/iucn/threat_stat...,Viburnum lantana L.
4,NaN,JBCLM:BGJBCLM:CM00002-08,NaN,https://www.gbif.org/species/3899518,Ziziphora aragonensis Pau,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN,Ziziphora aragonensis Pau
...,...,...,...,...,...,...,...,...,...,...,...
903,NaN,JBCLM:BGJBCLM:CM00686-11,NaN,https://www.gbif.org/species/8267849,Anarrhinum fruticosum Desf.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN,Anarrhinum fruticosum Desf.
904,NaN,JBCLM:BGJBCLM:CM00243-09,NaN,https://www.gbif.org/species/3085368,Agrostemma githago L.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN,Agrostemma githago L.
905,NaN,JBCLM:BGJBCLM:CM00591-11,NaN,https://www.gbif.org/species/5371711,Adonis vernalis L.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN,Adonis vernalis L.
906,NaN,JBCLM:BGJBCLM:CM00629-11,NaN,https://www.gbif.org/species/3742445,Anarrhinum laxiflorum Boiss.,NaN,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN,Anarrhinum laxiflorum Boiss.


Before concatenating the new rows with the WFO species, I have to split the original scientific names from GBIF into scientific name and authorhip, and the best, most reliable way to do so is through their species API, which I'll access using their pygbif library

In [14]:
import pandas as pd
from pygbif import species
import numpy as np

def get_full_authorship(scientific_name):
    # Handle empty, NaN, or non-string inputs
    if not isinstance(scientific_name, str) or not scientific_name.strip():
        return {"canonical_name": "N/A", "full_authorship": "N/A"}
    
    try:
        # Parse the name using GBIF
        result = species.name_parser(scientific_name)
        
        if not result or not result[0].get('parsed', False):
            return {"canonical_name": "N/A", "full_authorship": "N/A"}
        
        parsed_name = result[0]
        canonical_name = parsed_name.get('canonicalName', 'N/A')
        authorship = parsed_name.get('authorship', '')
        bracket_authorship = parsed_name.get('bracketAuthorship', '')
        
        # Construct full authorship
        if bracket_authorship:
            full_authorship = f"({bracket_authorship})"
            if authorship:
                full_authorship = f"{full_authorship} {authorship}"
        else:
            full_authorship = authorship if authorship else "N/A"
        
        return {
            "canonical_name": canonical_name,
            "full_authorship": full_authorship
        }
    except Exception as e:
        print(f"Error parsing {scientific_name}: {e}")
        return {"canonical_name": "N/A", "full_authorship": "N/A"}

# Process scientific names
germplasm_names = germplasm["scientific_name"].tolist()
name_only = []
author_only = []

for name in germplasm_names:
    result = get_full_authorship(name)
    name_only.append(result["canonical_name"])
    author_only.append(result["full_authorship"])

print(name_only)

['Ziziphora aragonensis', 'Vella pseudocytisus pseudocytisus', 'Zygophyllum fabago', 'Viburnum lantana', 'Ziziphora aragonensis', 'Ziziphora aragonensis', 'Viola cazorlensis', 'Ziziphora aragonensis', 'Viburnum tinus', 'Veronica chamaepithyoides', 'Thymus zygis sylvestris', 'Gypsophila vaccaria', 'Thymus zygis zygis', 'Veronica fruticans cantabrica', 'Thymus zygis sylvestris', 'Vella pseudocytisus paui', 'Thymus pulegioides', 'Thymus zygis zygis', 'Thymus vulgaris vulgaris', 'Thymus zygis gracilis', 'Thymus zygis sylvestris', 'Vaccinium myrtillus', 'Trollius europaeus', 'Thymus zygis gracilis', 'Trollius europaeus', 'Thymus zygis zygis', 'Vella pseudocytisus pseudocytisus', 'Thymus vulgaris vulgaris', 'Viburnum opulus', 'Thymus vulgaris aestivus', 'Thymus zygis gracilis', 'Thymus membranaceus', 'Thymus orospedanus', 'Viburnum lantana', 'Thymus moroderi', 'Thymus vulgaris vulgaris', 'Vella spinosa', 'Thymus membranaceus', 'Thymus pulegioides', 'Thymus leptophyllus leptophyllus', 'Vella 

In [15]:
germplasm = germplasm.drop(columns=["temp_sci_name"])
germplasm["scientific_name"] = name_only
germplasm["scientific_name_authorship"] = author_only
germplasm

,uniqid,accession_number,national_catalogue_code,taxon_identifier,scientific_name,scientific_name_authorship,vernacular_name,taxonomy_reference,determination_status,iucn_threat_category
0,NaN,JBCLM:BGJBCLM:CM00013-08,NaN,https://www.gbif.org/species/3899518,Ziziphora aragonensis,Pau,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
1,NaN,JBCLM:BGJBCLM:CM00009-08,NaN,https://www.gbif.org/species/7225676,Vella pseudocytisus pseudocytisus,N/A,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
2,NaN,JBCLM:BGJBCLM:CM00247-09,NaN,https://www.gbif.org/species/3189902,Zygophyllum fabago,L.,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
3,NaN,JBCLM:BGJBCLM:CM00854-13,NaN,https://www.gbif.org/species/2888615,Viburnum lantana,L.,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,http://rs.gbif.org/vocabulary/iucn/threat_stat...
4,NaN,JBCLM:BGJBCLM:CM00002-08,NaN,https://www.gbif.org/species/3899518,Ziziphora aragonensis,Pau,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
903,NaN,JBCLM:BGJBCLM:CM00686-11,NaN,https://www.gbif.org/species/8267849,Anarrhinum fruticosum,Desf.,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
904,NaN,JBCLM:BGJBCLM:CM00243-09,NaN,https://www.gbif.org/species/3085368,Agrostemma githago,L.,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
905,NaN,JBCLM:BGJBCLM:CM00591-11,NaN,https://www.gbif.org/species/5371711,Adonis vernalis,L.,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
906,NaN,JBCLM:BGJBCLM:CM00629-11,NaN,https://www.gbif.org/species/3742445,Anarrhinum laxiflorum,Boiss.,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN


Now I will concatenate the rows with the WFO information to the original dataset

In [16]:
germplasm2 = pd.concat([germplasm, merged_df], ignore_index = True)
germplasm2

,uniqid,accession_number,national_catalogue_code,taxon_identifier,scientific_name,scientific_name_authorship,vernacular_name,taxonomy_reference,determination_status,iucn_threat_category
0,NaN,JBCLM:BGJBCLM:CM00013-08,NaN,https://www.gbif.org/species/3899518,Ziziphora aragonensis,Pau,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
1,NaN,JBCLM:BGJBCLM:CM00009-08,NaN,https://www.gbif.org/species/7225676,Vella pseudocytisus pseudocytisus,N/A,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
2,NaN,JBCLM:BGJBCLM:CM00247-09,NaN,https://www.gbif.org/species/3189902,Zygophyllum fabago,L.,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
3,NaN,JBCLM:BGJBCLM:CM00854-13,NaN,https://www.gbif.org/species/2888615,Viburnum lantana,L.,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,http://rs.gbif.org/vocabulary/iucn/threat_stat...
4,NaN,JBCLM:BGJBCLM:CM00002-08,NaN,https://www.gbif.org/species/3899518,Ziziphora aragonensis,Pau,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
1810,NaN,JBCLM:BGJBCLM:CM00686-11,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Anarrhinum fruticosum,Desf.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
1811,NaN,JBCLM:BGJBCLM:CM00243-09,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Agrostemma githago,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
1812,NaN,JBCLM:BGJBCLM:CM00591-11,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Adonis vernalis,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
1813,NaN,JBCLM:BGJBCLM:CM00629-11,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Anarrhinum laxiflorum,Boiss.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN


In [20]:
from datetime import datetime
import time
lista = []
for i in range(1, len(germplasm2["uniqid"])+1):
    now = datetime.now()
    now = now.strftime('%Y%m%d%H%M%S%f')
    lista.append(now)
    time.sleep(0.001)
germplasm2["uniqid"] = lista
germplasm2

,uniqid,accession_number,national_catalogue_code,taxon_identifier,scientific_name,scientific_name_authorship,vernacular_name,taxonomy_reference,determination_status,iucn_threat_category
0,20251007160019885752,JBCLM:BGJBCLM:CM00013-08,NaN,https://www.gbif.org/species/3899518,Ziziphora aragonensis,Pau,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
1,20251007160019886954,JBCLM:BGJBCLM:CM00009-08,NaN,https://www.gbif.org/species/7225676,Vella pseudocytisus pseudocytisus,N/A,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
2,20251007160019888234,JBCLM:BGJBCLM:CM00247-09,NaN,https://www.gbif.org/species/3189902,Zygophyllum fabago,L.,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
3,20251007160019889901,JBCLM:BGJBCLM:CM00854-13,NaN,https://www.gbif.org/species/2888615,Viburnum lantana,L.,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,http://rs.gbif.org/vocabulary/iucn/threat_stat...
4,20251007160019891680,JBCLM:BGJBCLM:CM00002-08,NaN,https://www.gbif.org/species/3899518,Ziziphora aragonensis,Pau,NaN,GBIF Secretariat: GBIF Backbone Taxonomy. http...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
1810,20251007160022672771,JBCLM:BGJBCLM:CM00686-11,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Anarrhinum fruticosum,Desf.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
1811,20251007160022674376,JBCLM:BGJBCLM:CM00243-09,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Agrostemma githago,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
1812,20251007160022675811,JBCLM:BGJBCLM:CM00591-11,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Adonis vernalis,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
1813,20251007160022677297,JBCLM:BGJBCLM:CM00629-11,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Anarrhinum laxiflorum,Boiss.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN


In [24]:
germplasm2.to_csv("germplasm2.csv", index = False)

# Location

In [25]:
location = pd.DataFrame(columns=["uniqid", "accession_number", "national_catalogue_code", "acquisition_time", "collecting_time", "soil_type", "latitude", "longitude", "coordinate_uncertainty", "geodetic_datum", "maximum_elevation", "minimum_elevation", "terrain_inclination", "country_name", "country_2letter_code", "first_admin_subdivision", "second_admin_subdivision", "third_admin_subdivision", "fourth_admin_subdivision", "locality_description"])
location

,uniqid,accession_number,national_catalogue_code,acquisition_time,collecting_time,soil_type,latitude,longitude,coordinate_uncertainty,geodetic_datum,maximum_elevation,minimum_elevation,terrain_inclination,country_name,country_2letter_code,first_admin_subdivision,second_admin_subdivision,third_admin_subdivision,fourth_admin_subdivision,locality_description


In [26]:
location[["accession_number", "collecting_time", "latitude", "longitude", "coordinate_uncertainty", "geodetic_datum", "maximum_elevation", "country_name", "country_2letter_code", "first_admin_subdivision", "second_admin_subdivision", "third_admin_subdivision", "fourth_admin_subdivision", "locality_description"]] = data[["occurrenceID", "eventDate", "decimalLatitude", "decimalLongitude", "coordinateUncertaintyInMeters", "verbatimCoordinateSystem", "elevation","level0Name", "countryCode", "stateProvince", "county", "municipality", "locality", "verbatimLocality"]]
location["minimum_elevation"] = location["maximum_elevation"]
location.head()

,uniqid,accession_number,national_catalogue_code,acquisition_time,collecting_time,soil_type,latitude,longitude,coordinate_uncertainty,geodetic_datum,maximum_elevation,minimum_elevation,terrain_inclination,country_name,country_2letter_code,first_admin_subdivision,second_admin_subdivision,third_admin_subdivision,fourth_admin_subdivision,locality_description
0,NaN,JBCLM:BGJBCLM:CM00013-08,NaN,NaN,2001-06-26,NaN,39.0700,-2.670000,NaN,NaN,NaN,NaN,NaN,Spain,ES,Albacete,NaN,NaN,Villarrobledo,NaN
1,NaN,JBCLM:BGJBCLM:CM00009-08,NaN,NaN,2006-07-14,NaN,39.9997,-3.529319,NaN,NaN,NaN,NaN,NaN,Spain,ES,Toledo,NaN,NaN,Ontigola,NaN
2,NaN,JBCLM:BGJBCLM:CM00247-09,NaN,NaN,2009-07-03,NaN,38.4900,-1.670000,NaN,NaN,NaN,NaN,NaN,Spain,ES,Albacete,NaN,NaN,Hellín,NaN
3,NaN,JBCLM:BGJBCLM:CM00854-13,NaN,NaN,2013-09-04,NaN,42.3218,-0.334900,NaN,NaN,NaN,NaN,NaN,Spain,ES,Huesca,NaN,NaN,Jaca,NaN
4,NaN,JBCLM:BGJBCLM:CM00002-08,NaN,NaN,1997-07-29,NaN,39.1500,-2.490000,NaN,NaN,NaN,NaN,NaN,Spain,ES,Albacete,NaN,NaN,Villarrobledo,NaN


In [27]:
from datetime import datetime
import time
lista = []
for i in range(1, len(location["uniqid"])+1):
    now = datetime.now()
    now = now.strftime('%Y%m%d%H%M%S%f')
    lista.append(now)
    time.sleep(0.001)
location["uniqid"] = lista
location

,uniqid,accession_number,national_catalogue_code,acquisition_time,collecting_time,soil_type,latitude,longitude,coordinate_uncertainty,geodetic_datum,maximum_elevation,minimum_elevation,terrain_inclination,country_name,country_2letter_code,first_admin_subdivision,second_admin_subdivision,third_admin_subdivision,fourth_admin_subdivision,locality_description
0,20251007160058109185,JBCLM:BGJBCLM:CM00013-08,NaN,NaN,2001-06-26,NaN,39.070000,-2.670000,NaN,NaN,NaN,NaN,NaN,Spain,ES,Albacete,NaN,NaN,Villarrobledo,NaN
1,20251007160058118840,JBCLM:BGJBCLM:CM00009-08,NaN,NaN,2006-07-14,NaN,39.999700,-3.529319,NaN,NaN,NaN,NaN,NaN,Spain,ES,Toledo,NaN,NaN,Ontigola,NaN
2,20251007160058120379,JBCLM:BGJBCLM:CM00247-09,NaN,NaN,2009-07-03,NaN,38.490000,-1.670000,NaN,NaN,NaN,NaN,NaN,Spain,ES,Albacete,NaN,NaN,Hellín,NaN
3,20251007160058121814,JBCLM:BGJBCLM:CM00854-13,NaN,NaN,2013-09-04,NaN,42.321800,-0.334900,NaN,NaN,NaN,NaN,NaN,Spain,ES,Huesca,NaN,NaN,Jaca,NaN
4,20251007160058131104,JBCLM:BGJBCLM:CM00002-08,NaN,NaN,1997-07-29,NaN,39.150000,-2.490000,NaN,NaN,NaN,NaN,NaN,Spain,ES,Albacete,NaN,NaN,Villarrobledo,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
903,20251007160059560666,JBCLM:BGJBCLM:CM00686-11,NaN,NaN,2011-06-28,NaN,38.263100,-0.799110,NaN,NaN,NaN,NaN,NaN,Spain,ES,Alicante,NaN,NaN,Crevillente,NaN
904,20251007160059562253,JBCLM:BGJBCLM:CM00243-09,NaN,NaN,2009-06-19,NaN,38.723359,-1.770983,NaN,NaN,NaN,NaN,NaN,Spain,ES,Albacete,NaN,NaN,Albacete,NaN
905,20251007160059563659,JBCLM:BGJBCLM:CM00591-11,NaN,NaN,2011-06-05,NaN,38.656240,-1.860258,NaN,NaN,NaN,NaN,NaN,Spain,ES,Albacete,NaN,NaN,Albacete,NaN
906,20251007160059565264,JBCLM:BGJBCLM:CM00629-11,NaN,NaN,2011-08-23,NaN,38.659700,-2.390000,NaN,NaN,NaN,NaN,NaN,Spain,ES,Albacete,NaN,NaN,Peñascosa,NaN


In [29]:
location["latitude"]

0      39.070000
1      39.999700
2      38.490000
3      42.321800
4      39.150000
         ...    
903    38.263100
904    38.723359
905    38.656240
906    38.659700
907    38.070000
Name: latitude, Length: 908, dtype: float64

In [28]:
location.to_csv("location.csv", index = False)